# R3b deferred-diagnostic training entry

Default auto: source/CPU evidence, permission/data preflight, then HOLD.
It never repeats the accepted Pilot or starts unapproved 500-step training.
After explicit release, training records lightweight telemetry and all original
event/checkpoint commits; independent diagnostics use analysis-notebook.ipynb.


In [ ]:
EXPERIMENT = "CLM-001-R3b"
ACTION = "auto"  # auto | preflight | train | resume | evaluate | publish-only
SEGMENT_END_STEP = 200  # formal approved segments:200→350→500; doesn't change training.steps
HF_REPO = "libingjiang/minicells-research-checkpoints"
RUN_ID = None  # program resolves immutable identity; never generate a new one on resume
CHECKPOINT_REF = None  # optional exact GitHub checkpoint-index URL; otherwise HF lookup

In [ ]:
from pathlib import Path
import importlib.util
import json
import os
import subprocess
import sys
from kaggle_secrets import UserSecretsClient

SOURCE_SHA = "c29d2948610547d0cafa64acefb0a5d8a2e70577"
CODE = Path("/kaggle/working/minicells-r3b-code-" + SOURCE_SHA[:12])
WORK = Path("/kaggle/working/minicells-r3b-" + SOURCE_SHA[:12])
ENTRY = (
    CODE
    / "research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3b/run.py"
)
os.environ.update(
    OMP_NUM_THREADS="1",
    MKL_NUM_THREADS="1",
    TOKENIZERS_PARALLELISM="false",
    HF_HUB_DISABLE_PROGRESS_BARS="1",
)


def execute(args, cwd=None):
    print("[R3b] " + " ".join(map(str, args)), flush=True)
    return subprocess.run(list(map(str, args)), cwd=cwd, check=True)


secrets = UserSecretsClient()
for name in ("HF_TOKEN", "GITHUB_TOKEN"):
    if not os.environ.get(name, "").strip():
        try:
            os.environ[name] = secrets.get_secret(name).strip()
        except Exception:
            raise RuntimeError("Enable existing Kaggle Secret " + name) from None
    if not os.environ[name]:
        raise RuntimeError("Missing existing Kaggle Secret " + name)

if not (CODE / ".git").exists():
    CODE.mkdir(parents=True, exist_ok=False)
    execute(["git", "init", "--quiet", CODE])
    execute(["git", "remote", "add", "origin", "https://github.com/ArcheLabs/mini-cells.git"], CODE)
    execute(["git", "sparse-checkout", "init", "--no-cone"], CODE)
    execute(
        [
            "git",
            "sparse-checkout",
            "set",
            "--no-cone",
            "/pyproject.toml",
            "/src/",
            "/shared/python/",
            "/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/",
            "/research/projects/cellular-language-models/engineering/kaggle-continuation/",
            "/research/projects/cellular-language-models/experiments/006-path-first-clm/",
            "/research/projects/cellular-language-models/experiments/005-compositional-cell-graph/implementation/",
            "/research/projects/continual-learning/context/04-continual-learning/scripts/research/",
            "/research/projects/cellular-language-models/context/08-hybrid-clm/scripts/research/",
        ],
        CODE,
    )
    execute(["git", "fetch", "--depth=1", "--filter=blob:none", "origin", SOURCE_SHA], CODE)
    execute(["git", "checkout", "--detach", "FETCH_HEAD"], CODE)
if subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=CODE, text=True).strip() != SOURCE_SHA:
    raise RuntimeError("SOURCE_SHA_MISMATCH")

# Install only missing data/storage packages. Torch/numpy are supplied by Kaggle.
requirements = {
    "datasets": "datasets>=2.18,<5",
    "tokenizers": "tokenizers",
    "huggingface_hub": "huggingface_hub>=0.23",
}
missing = [
    requirement
    for module, requirement in requirements.items()
    if importlib.util.find_spec(module) is None
]
if missing:
    execute([sys.executable, "-m", "pip", "install", "--quiet", *missing])
args = [
    sys.executable,
    ENTRY,
    "--mode",
    ACTION,
    "--output",
    WORK,
    "--hf-repo",
    HF_REPO,
    "--segment-end-step",
    str(SEGMENT_END_STEP),
    "--publish-github",
]
if RUN_ID is not None:
    args += ["--run-id", RUN_ID]
if CHECKPOINT_REF is not None:
    args += ["--checkpoint-ref", CHECKPOINT_REF]
print(
    json.dumps(
        {
            "EXPERIMENT": EXPERIMENT,
            "SOURCE_SHA": SOURCE_SHA,
            "formal_500_step_authorized": False,
            "ACTION": ACTION,
        }
    ),
    flush=True,
)
execute(args, CODE)